# AstroCLIMB zero-shot — Qwen3-VL-4B-Instruct

This notebook is an inference-only version of the Run 0 pipeline for `Qwen/Qwen3-VL-4B-Instruct`. It verifies CSV reading, modality detection, base64 image decoding, multimodal chat construction, 4-bit model loading, constrained predictions, macro-F1, and submission generation. It performs no training or parameter updates. NF4 quantization is used only to reduce inference memory.

Before running: select the Kaggle **GPU T4 x2** accelerator and enable Internet, or attach the model as a Kaggle dataset. The notebook intentionally exposes one T4 so its behavior stays aligned with the Run 0 check. Do not attach or read `solution.csv`.


In [ ]:
# Install only the inference packages. Keep Kaggle's torch, torchvision, Pillow and scikit-learn unchanged.
%pip install -q --upgrade --upgrade-strategy only-if-needed "transformers==4.57.1" "accelerate==1.10.1" "bitsandbytes==0.47.0"


In [ ]:
import base64
import csv
import io
import os
import random
import sys
import time
from pathlib import Path

# Keep the Run 0 execution shape: one visible T4 per notebook process.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

ImageFile.LOAD_TRUNCATED_IMAGES = True
csv.field_size_limit(sys.maxsize)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Visible GPU count:', torch.cuda.device_count(), '(this notebook intentionally exposes one T4)')
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1), 'GiB')
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Notebook settings.'
assert torch.cuda.device_count() == 1, 'This notebook expects one visible GPU. Restart the Kaggle session and run from the first cell.'
assert torch.cuda.get_device_capability(0)[0] == 7, 'This notebook is configured for a T4-class GPU.'


## Configuration

`PER_CLASS=64` gives a balanced 256-example zero-shot validation sample. The labeled CSV is ordered in class blocks, so the reader may need to stream past row 7,000 to reach `unrelated_papers`; it retains only the selected examples in memory. The default 448×448 area budget is intentionally conservative.


In [ ]:
MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
MODEL_HINT_GROUPS = [['qwen3'], ['4b']]
MODEL_SLUG = 'qwen3vl4b'
PER_CLASS = 64
MAX_ROWS_TO_SCAN = 8000
MIN_PIXELS = 256 * 256
MAX_PIXELS = 448 * 448
MAX_TEXT_CHARS_PER_OBJECT = 3000
USE_SWAP_TTA = False  # Turn on only after ordinary validation works.

TARGET_COLUMNS = [
    'same_figure',
    'same_paper',
    'related_papers',
    'unrelated_papers',
]
LABEL_TO_DIGIT = {name: str(i) for i, name in enumerate(TARGET_COLUMNS)}
DIGIT_TO_LABEL = {i: name for i, name in enumerate(TARGET_COLUMNS)}

def find_train_csv():
    preferred = Path('/kaggle/input/astroclimb/train.csv')
    if preferred.exists():
        return preferred
    candidates = list(Path('/kaggle/input').rglob('train.csv')) if Path('/kaggle/input').exists() else []
    candidates += list(Path('data').glob('train.csv'))
    candidates = sorted(candidates, key=lambda p: ('astroclimb' not in str(p).lower(), len(str(p))))
    if not candidates:
        raise FileNotFoundError('train.csv not found. Attach the AstroCLIMB competition data first.')
    return candidates[0]

def find_model_path():
    # Prefer an attached Kaggle model/dataset when Internet is disabled.
    if Path('/kaggle/input').exists():
        configs = list(Path('/kaggle/input').rglob('config.json'))
        model_dirs = [
            p.parent for p in configs
            if 'vl' in str(p).lower()
            and all(any(hint in str(p).lower() for hint in group) for group in MODEL_HINT_GROUPS)
        ]
        if model_dirs:
            return str(sorted(model_dirs, key=lambda p: len(str(p)))[0])
    return MODEL_ID

TRAIN_CSV = find_train_csv()
MODEL_PATH = find_model_path()
print('Train CSV:', TRAIN_CSV)
print('Model:', MODEL_PATH)


## Stream a small balanced validation sample

The CSV contains very large base64 fields and is ordered by label. This cell streams until it has reached all four label blocks and collected the requested quota. It may read through roughly 7,000 rows, but it never loads the full 10 GB file into pandas and retains only the selected examples.


In [ ]:
def row_label(row):
    values = np.array([int(float(row[c])) for c in TARGET_COLUMNS])
    if values.sum() != 1:
        return None
    return int(values.argmax())

def looks_like_image(value):
    if not isinstance(value, str):
        return False
    value = value.lstrip()
    if value.startswith('data:image'):
        return True
    return value.startswith(('iVBORw0KGgo', '/9j/', 'UklGR', 'R0lGOD'))

def modality(row):
    a, b = looks_like_image(row['obj_1']), looks_like_image(row['obj_2'])
    return 'II' if a and b else ('IT' if a or b else 'TT')

def load_balanced_validation(path, per_class=64, max_rows=1500):
    buckets = {i: [] for i in range(4)}
    scanned = 0
    with open(path, 'r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        missing = {'obj_1', 'obj_2', *TARGET_COLUMNS} - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'Missing expected columns: {sorted(missing)}')
        for row in reader:
            scanned += 1
            y = row_label(row)
            if y is not None and len(buckets[y]) < per_class:
                buckets[y].append({
                    'id': row.get('id', str(scanned - 1)),
                    'obj_1': row['obj_1'],
                    'obj_2': row['obj_2'],
                    'label': y,
                })
            if all(len(v) >= per_class for v in buckets.values()) or scanned >= max_rows:
                break
    rows = [item for bucket in buckets.values() for item in bucket]
    random.Random(SEED).shuffle(rows)
    print('Rows scanned:', scanned)
    print('Collected:', {DIGIT_TO_LABEL[k]: len(v) for k, v in buckets.items()})
    if min(map(len, buckets.values())) < 8:
        raise RuntimeError('Too few examples in at least one class. Increase MAX_ROWS_TO_SCAN.')
    return pd.DataFrame(rows)

validation_sample_started = time.perf_counter()
val_df = load_balanced_validation(TRAIN_CSV, PER_CLASS, MAX_ROWS_TO_SCAN)
validation_sample_seconds = time.perf_counter() - validation_sample_started
print(f'Sample collection time: {validation_sample_seconds:.1f}s ({validation_sample_seconds / 60:.2f} min)')
val_df['modality'] = val_df.apply(modality, axis=1)
display(pd.crosstab(val_df['label'].map(DIGIT_TO_LABEL), val_df['modality'], margins=True))

val_df = val_df.reset_index(drop=True)
print('Validation:', len(val_df))


## Decode objects and construct conversations

In [ ]:
SYSTEM_PROMPT = '''You classify the relationship between two objects from astronomy papers.
0: The objects are the figure and caption of the same scientific figure.
1: The objects are from different figures in the same paper.
2: The objects are from different papers and one paper cites the other.
3: The objects are from unrelated papers.
The relationship is symmetric. Output only one digit: 0, 1, 2, or 3.'''.strip()

def decode_image(value):
    value = value.strip()
    if value.startswith('data:image'):
        value = value.split(',', 1)[1]
    raw = base64.b64decode(value, validate=False)
    image = Image.open(io.BytesIO(raw))
    image.load()
    return image.convert('RGB')

def shorten_caption(text, max_chars=MAX_TEXT_CHARS_PER_OBJECT):
    text = str(text)
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    return text[:half] + '\n[...middle truncated...]\n' + text[-half:]

def object_content(number, value):
    if looks_like_image(value):
        return [
            {'type': 'text', 'text': f'Object {number} is a scientific figure:'},
            {'type': 'image', 'image': decode_image(value)},
        ]
    return [{
        'type': 'text',
        'text': f'Object {number} is a figure caption:\n{shorten_caption(value)}',
    }]

def build_messages(row, include_answer=True, swap=False):
    obj_1, obj_2 = row['obj_1'], row['obj_2']
    if swap:
        obj_1, obj_2 = obj_2, obj_1
    content = object_content(1, obj_1) + object_content(2, obj_2)
    content.append({'type': 'text', 'text': 'Classify their relationship. Reply with one digit only.'})
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': content},
    ]
    if include_answer:
        messages.append({'role': 'assistant', 'content': [{'type': 'text', 'text': str(int(row['label']))}]})
    return messages

# Decode one example of each available modality before loading the model.
for mode in sorted(val_df['modality'].unique()):
    row = val_df[val_df['modality'] == mode].iloc[0]
    messages = build_messages(row, include_answer=True)
    image_sizes = [item['image'].size for msg in messages for item in msg['content'] if item.get('type') == 'image']
    print(mode, 'label=', row['label'], 'image sizes=', image_sizes)

## Load Qwen3-VL-4B-Instruct for 4-bit inference

T4 uses FP16. SDPA is selected because FlashAttention 2 is not a suitable default for Turing/T4 GPUs. The model is placed only on `cuda:0`. Quantization is used strictly for zero-shot inference.


In [ ]:
model_load_started = time.perf_counter()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = 'right'

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    quantization_config=quantization_config,
    dtype=torch.float16,
    attn_implementation='sdpa',
    device_map={'': 0},
)
model.eval()
model.config.use_cache = True

label_token_ids = []
for digit in '0123':
    ids = processor.tokenizer.encode(digit, add_special_tokens=False)
    assert len(ids) == 1, f'Label {digit!r} is not one token: {ids}'
    label_token_ids.append(ids[0])
print('Label token IDs:', label_token_ids)

torch.cuda.synchronize()
model_load_seconds = time.perf_counter() - model_load_started
print('Allocated GiB:', round(torch.cuda.memory_allocated() / 2**30, 2))
print(f'Model loading time: {model_load_seconds:.1f}s ({model_load_seconds / 60:.2f} min)')


## Constrained validation

This uses one forward pass and compares only the next-token logits for the four class digits. Set `USE_SWAP_TTA=True` later to average both object orders.

In [ ]:
@torch.inference_mode()
def predict_probabilities(row, swap=False):
    messages = build_messages(row, include_answer=False, swap=swap)
    batch = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors='pt',
    )
    batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
    logits = model(**batch).logits[0, -1, label_token_ids].float()
    return torch.softmax(logits, dim=-1).cpu().numpy()

model.eval()
model.config.use_cache = True
torch.cuda.synchronize()
validation_started = time.perf_counter()
probabilities = []
for index, row in enumerate(val_df.to_dict('records')):
    p = predict_probabilities(row, swap=False)
    if USE_SWAP_TTA:
        p = 0.5 * (p + predict_probabilities(row, swap=True))
    probabilities.append(p)
    if (index + 1) % 10 == 0:
        print(f'Validated {index + 1}/{len(val_df)}')

torch.cuda.synchronize()
validation_seconds = time.perf_counter() - validation_started
validation_seconds_per_row = validation_seconds / max(1, len(val_df))
print(f'Validation inference time: {validation_seconds:.1f}s ({validation_seconds / 60:.2f} min)')
print(f'Inference seconds/row: {validation_seconds_per_row:.3f}s')
print(f'Projected time for 10,000 test rows: {validation_seconds_per_row * 10000 / 3600:.2f} hours')
probabilities = np.stack(probabilities)
y_true = val_df['label'].to_numpy()
y_pred = probabilities.argmax(axis=1)

print('Macro-F1:', f1_score(y_true, y_pred, average='macro'))
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(4)),
    target_names=TARGET_COLUMNS,
    digits=4,
    zero_division=0,
))
display(pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=list(range(4))),
    index=[f'true_{x}' for x in TARGET_COLUMNS],
    columns=[f'pred_{x}' for x in TARGET_COLUMNS],
))

results = val_df[['id', 'label', 'modality']].copy()
results['prediction'] = y_pred
for i, name in enumerate(TARGET_COLUMNS):
    results[f'p_{name}'] = probabilities[:, i]
results.to_csv(f'{MODEL_SLUG}_validation_predictions.csv', index=False)
display(results.head())

## Stream the complete test set and create `submission.csv`

The test CSV is streamed one row at a time, so the base64 objects never accumulate in RAM. The competition submission is one-hot encoded. Raw class probabilities are also saved for later bias tuning or ensembling. `USE_SWAP_TTA=False` is recommended initially because swapped inference doubles runtime.

Set `TEST_LIMIT=32` for a quick file-format test; leave it as `None` to produce the complete Kaggle submission.


In [ ]:
TEST_LIMIT = None  # Use 32 to test the inference/output pipeline; None processes all test rows.
TEST_LOG_EVERY = 100
EXPECTED_TEST_ROWS = 10000
FAIL_FAST = True

def find_test_csv():
    preferred = Path('/kaggle/input/astroclimb/test.csv')
    if preferred.exists():
        return preferred
    candidates = list(Path('/kaggle/input').rglob('test.csv')) if Path('/kaggle/input').exists() else []
    candidates += list(Path('data').glob('test.csv'))
    candidates = sorted(candidates, key=lambda p: ('astroclimb' not in str(p).lower(), len(str(p))))
    if not candidates:
        raise FileNotFoundError('test.csv not found. Attach the AstroCLIMB competition data first.')
    return candidates[0]

TEST_CSV = find_test_csv()
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
SUBMISSION_PATH = OUTPUT_ROOT / 'submission.csv'
PROBABILITY_PATH = OUTPUT_ROOT / 'submission_probabilities.csv'
ERROR_PATH = OUTPUT_ROOT / 'test_inference_errors.csv'
print('Test CSV:', TEST_CSV)
print('Submission:', SUBMISSION_PATH)

In [ ]:
model.eval()
model.config.use_cache = True
torch.cuda.synchronize()
test_started = time.perf_counter()
test_counts = np.zeros(4, dtype=np.int64)
errors = []
rows_written = 0

with (
    open(TEST_CSV, 'r', encoding='utf-8', newline='') as source,
    open(SUBMISSION_PATH, 'w', encoding='utf-8', newline='') as submission_file,
    open(PROBABILITY_PATH, 'w', encoding='utf-8', newline='') as probability_file,
):
    reader = csv.DictReader(source)
    missing = {'id', 'obj_1', 'obj_2'} - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f'Test CSV is missing columns: {sorted(missing)}')

    submission_writer = csv.DictWriter(
        submission_file,
        fieldnames=['id', *TARGET_COLUMNS],
    )
    probability_writer = csv.DictWriter(
        probability_file,
        fieldnames=['id', *[f'p_{name}' for name in TARGET_COLUMNS]],
    )
    submission_writer.writeheader()
    probability_writer.writeheader()

    for row_index, row in enumerate(reader):
        if TEST_LIMIT is not None and row_index >= TEST_LIMIT:
            break
        try:
            p = predict_probabilities(row, swap=False)
            if USE_SWAP_TTA:
                p = 0.5 * (p + predict_probabilities(row, swap=True))
            prediction = int(np.argmax(p))
        except Exception as exc:
            errors.append({'id': row.get('id', row_index), 'error': repr(exc)})
            if FAIL_FAST:
                raise
            # Keep the output structurally complete if explicitly running non-fail-fast.
            p = np.array([0.0, 0.0, 0.0, 1.0], dtype=np.float32)
            prediction = 3

        submission_writer.writerow({
            'id': row['id'],
            **{name: int(i == prediction) for i, name in enumerate(TARGET_COLUMNS)},
        })
        probability_writer.writerow({
            'id': row['id'],
            **{f'p_{name}': float(p[i]) for i, name in enumerate(TARGET_COLUMNS)},
        })
        test_counts[prediction] += 1
        rows_written += 1

        if rows_written % TEST_LOG_EVERY == 0:
            submission_file.flush()
            probability_file.flush()
            elapsed = time.perf_counter() - test_started
            seconds_per_row = elapsed / rows_written
            target_rows = TEST_LIMIT if TEST_LIMIT is not None else EXPECTED_TEST_ROWS
            remaining = max(0, target_rows - rows_written)
            eta_seconds = seconds_per_row * remaining
            print(
                f'Predicted {rows_written}/{target_rows} | '
                f'elapsed={elapsed / 60:.1f} min | '
                f'{seconds_per_row:.3f}s/row | ETA={eta_seconds / 3600:.2f}h | '
                f'class counts={test_counts.tolist()}'
            )

if errors:
    pd.DataFrame(errors).to_csv(ERROR_PATH, index=False)

torch.cuda.synchronize()
test_seconds = time.perf_counter() - test_started
print('Finished test inference.')
print('Rows written:', rows_written)
print(f'Test inference wall time: {test_seconds:.1f}s ({test_seconds / 3600:.2f}h)')
print(f'Test inference seconds/row: {test_seconds / max(1, rows_written):.3f}s')
print('Prediction counts:', dict(zip(TARGET_COLUMNS, test_counts.tolist())))
print('Errors:', len(errors))

In [ ]:
# Validate the exact file that will be submitted.
submission_check = pd.read_csv(SUBMISSION_PATH)
expected_columns = ['id', *TARGET_COLUMNS]
assert submission_check.columns.tolist() == expected_columns
assert len(submission_check) == rows_written
assert submission_check['id'].is_unique
assert submission_check[TARGET_COLUMNS].isin([0, 1]).all().all()
assert (submission_check[TARGET_COLUMNS].sum(axis=1) == 1).all()
assert not submission_check.isna().any().any()

if TEST_LIMIT is not None:
    print(f'WARNING: TEST_LIMIT={TEST_LIMIT}; this is not a complete Kaggle submission.')
else:
    print('Complete submission is ready:', SUBMISSION_PATH)
display(submission_check.head())
display(submission_check[TARGET_COLUMNS].sum().rename('prediction_count').to_frame())

## Next steps

Compare macro-F1, per-class recall, the confusion matrix, and inference time across the three model notebooks. Select the model only from validation results, then run its complete test inference with `TEST_LIMIT=None` and submit the validated `submission.csv`.
